# 📄 Extraction automatique des interdictions de vente — PDFs Société Générale

Ce notebook extrait automatiquement les **statuts d'interdiction de vente** depuis des Final Terms en format PDF.  
Il produit un tableau structuré indiquant, pour chaque document, si la vente est interdite (**`True`**) ou autorisée (**`False`**) selon trois zones géographiques et le type d'investisseur ciblé.

---

## Contexte

Les Final Terms contiennent une section standardisée sur les restrictions de vente (« Interdiction de Ventes » / « Prohibition of Sales » / « Verkaufsverbot »).  
Cette section liste, pour deux profils d'investisseurs et trois zones géographiques, si la prohibition s'applique ou non.

Ce notebook cible spécifiquement la paire **« Personnes Non Clientèle de Détail »** (investisseurs non professionnels), pour les zones :

| Colonne | Zone | Termes reconnus |
|---|---|---|
| `prohibition to EEA` | Espace Économique Européen | EEE / EEA / EWR |
| `prohibition to UK` | Royaume-Uni | UK / United Kingdom / Royaume-Uni / Königreich |
| `prohibition to Swiss` | Suisse | Suisse / Swiss / Schweiz |

---

## Langues supportées

| Langue | Marqueur de section | Marqueur d'interdiction |
|---|---|---|
| 🇫🇷 Français | `9. PLACEMENT` | `Interdiction de Ventes` |
| 🇬🇧 Anglais | `9. DISTRIBUTION` | `Prohibition of Sales` |
| 🇩🇪 Allemand | `9. PLATZIERUNG` | `Verkaufsverbot` |

> **Remarque :** le numéro de section (9, 8…) est détecté automatiquement — il peut varier selon les documents.

---

## Structure du notebook

| Cellule | Rôle |
|---|---|
| **Cellule 1** | Chargement des librairies, paramètres et fonctions de traitement |
| **Cellule 2** | Boucle de traitement, affichage du résultat et export CSV |

---

## Prérequis

```bash
pip install pdfplumber pandas tqdm
```

Les PDFs à analyser doivent être placés dans le dossier **`pdf_to_extract/`** (à créer s'il n'existe pas).

## Cellule 1 — Chargement & fonctions de traitement

Cette cellule charge les librairies et définit toutes les fonctions utilisées dans le pipeline.  
**Elle doit être exécutée en premier**, avant la cellule de traitement.

### Pipeline de traitement appliqué à chaque PDF

```
PDF
 │
 ├─ 1. Extraction du texte brut (pdfplumber)
 │
 ├─ 2. Suppression du bruit connu ("DEFINITIVES APPLICABLES", "APPLICABLE FINAL"…)
 │
 ├─ 3. Isolation de la section pertinente
 │       → entre "N. PLACEMENT / DISTRIBUTION / PLATZIERUNG"
 │         et   "N. MODALITÉS DE L'OFFRE / TERMS AND CONDITIONS…"
 │
 ├─ 4. Découpage à partir du premier marqueur d'interdiction
 │       → "Interdiction de Ventes" / "Prohibition of Sales" / "Verkaufsverbot"
 │       → détecte aussi la langue du document
 │
 ├─ 5. Nettoyage
 │       → suppression des lignes entièrement en MAJUSCULES (titres parasites)
 │       → troncature après la dernière ligne contenant un mot-clé pertinent
 │         (évite d'embarquer la section "intermédiaire" qui suit les blocs)
 │
 └─ 6. Extraction des statuts par bloc
         → chaque bloc = une ligne de statut + ses lignes de contexte
         → identification du type de personne (personne / investisseur)
         → identification de la zone géographique (EEE, UK, Suisse)
         → si la zone est absente du bloc, elle est inférée du bloc précédent
         → résolution finale : True / False / Error
```

### Valeurs possibles dans le résultat

| Valeur | Signification |
|---|---|
| `True` | La prohibition **s'applique** (vente interdite) |
| `False` | La prohibition **ne s'applique pas** (vente autorisée) |
| `Error` | Statut introuvable, ambigu, ou section non reconnue dans ce PDF |

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
# !pip install -U pdfplumber pandas

import pdfplumber
import re
import unicodedata
from pathlib import Path
import pandas as pd

PDF_FOLDER = Path("pdf_to_extract")  # dossier contenant les PDFs à traiter

# ── Normalisation ────────────────────────────────────────────────────────────
def _normalize(s: str) -> str:
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = re.sub(r"[\u0027\u2018\u2019\u201a\u2032]", "'", s)
    s = re.sub(r"\s+", " ", s)
    return s.casefold().strip()

# ── Lignes full-caps ─────────────────────────────────────────────────────────
def is_full_caps_line(line: str) -> bool:
    letters = [ch for ch in line if ch.isalpha()]
    return len(letters) >= 2 and all(ch.isupper() for ch in letters)

# ── Statuts → booléen ────────────────────────────────────────────────────────
ALLOWED_STATUSES = {
    "applicable",
    "non applicable",
    "not applicable",
    "sans objet",
    "nicht anwendbar",
    "anwendbar",
}

# True  = la prohibition s'applique
# False = la prohibition ne s'applique pas
STATUS_TO_BOOL = {
    "applicable"     : True,
    "anwendbar"      : True,
    "non applicable" : False,
    "not applicable" : False,
    "sans objet"     : False,
    "nicht anwendbar": False,
}

def extract_status(line: str) -> str | None:
    n = _normalize(line)
    # Trier par longueur décroissante : "non applicable" avant "applicable"
    for status in sorted(ALLOWED_STATUSES, key=len, reverse=True):
        if n.endswith(status):
            return status
    return None

# ── Type de personne (FR | EN | DE) ──────────────────────────────────────────
# personne EN PREMIER : évite que "Retail" dans investisseur écrase "Non Retail Clients"
PERSON_TYPES = [
    ("personne",     re.compile(r"Personnes?\b|Persons?\b|Non.Natural|Non.Retail|Clients?\b|Nat[u\u00fc]rliche\b", re.IGNORECASE)),
    ("investisseur", re.compile(r"Investisseurs?|Investors?\b|Privatinvestoren|Retail\b",   re.IGNORECASE)),
]

# ── Zone géographique (FR | EN | DE) ─────────────────────────────────────────
ZONES = [
    ("EEE",         re.compile(r"\bEEE\b|\bEEA\b|\bEWR\b",                               re.IGNORECASE)),
    ("Royaume-Uni", re.compile(r"Royaume.Uni|United\s+Kingdom|\bUK\b|K[o\u00f6]nigreich",  re.IGNORECASE)),
    ("Suisse",      re.compile(r"\bSuisse\b|\bSwiss\b|\bSchweiz\b",                       re.IGNORECASE)),
]

# ── Regex multilingues ───────────────────────────────────────────────────────
cleanup_re = re.compile(
    r"DEFINITIVES\s+APPLICABLES"
    r"|APPLICABLE\s+FINAL"
    r"|ANWENDBARE\s+ENDG[U\u00dc]LTIGE",
    re.IGNORECASE,
)
# \d+ rend le numéro de section flexible (certains docs ont 8. DISTRIBUTION / 9. TERMS…)
start_re = re.compile(
    r"(?m)^\s*\d+\s*\.\s*(?:PLACEMENT|DISTRIBUTION|PLATZIERUNG)\b",
    re.IGNORECASE,
)
end_re = re.compile(
    r"(?m)^\s*\d+\s*\.\s*(?:"
    r"MODALIT(?:ES|\u00c9S)\s+DE\s+L[\u0027\u2018\u2019]OFFRE"
    r"|TERMS\s+AND\s+CONDITIONS\s+OF\s+THE\s+OFFER"
    r"|EMISSIONSBEDINGUNGEN\s+DES\s+ANGEBOTS"
    r")",
    re.IGNORECASE,
)
interdiction_re = re.compile(
    r"(?:Interdiction\s+de\s+Ventes|Prohibition\s+of\s+Sales|Verkaufsverbot)\b",
    re.IGNORECASE,
)

# ── Détection de la langue ───────────────────────────────────────────────────
def detect_language(m_int_match: re.Match) -> str:
    word = m_int_match.group(0).lower()
    if "interdiction" in word: return "French"
    if "prohibition"  in word: return "English"
    if "verkaufsverbot" in word: return "German"
    return "Unknown"

# ── Parsing des blocs → {title: status} ──────────────────────────────────────
# ── Troncature après le dernier mot-clé pertinent ────────────────────────────
# Évite d'embarquer la section "intermédiaire" qui suit les blocs d'interdiction
MEANINGFUL_RE = re.compile(
    r"Interdiction\s+de\s+Ventes|Prohibition\s+of\s+Sales|Verkaufsverbot"
    r"|Investisseurs?|Investors?\b|Privatinvestoren"
    r"|Personnes?\b|Persons?\b|Non[\s\-]Natural|Non[\s\-]Retail|Clients?\b|Nat[u\u00fc]rliche\b"
    r"|D[\u00e9e]tail\b"
    r"|\bEEE\b|\bEEA\b|\bEWR\b|\bUK\b|United\s+Kingdom|Royaume.Uni|K[o\u00f6]nigreich"
    r"|\bSuisse\b|\bSwiss\b|\bSchweiz\b"
    r"|\bApplicable\b|\bAnwendbar\b|Non\s+Applicable|Not\s+Applicable|Nicht\s+Anwendbar|Sans\s+Objet",
    re.IGNORECASE,
)

def trim_to_last_meaningful(text: str) -> str:
    """Supprime les lignes de queue sans lien avec les blocs d'interdiction."""
    lines = text.splitlines()
    last_idx = 0
    for i, line in enumerate(lines):
        if MEANINGFUL_RE.search(line):
            last_idx = i
    return "\n".join(lines[: last_idx + 1]).strip()

def parse_blocks(text: str) -> dict[str, list[str]]:
    """
    Retourne un dict title → liste de statuts trouvés.
    Une liste de longueur > 1 signale une ambiguïté (→ Error dans la colonne).
    Zone inférée du bloc précédent quand absente (cas PDF allemands).
    """
    lines = text.splitlines()
    status_positions = [i for i, l in enumerate(lines) if extract_status(l) is not None]

    tuples: list[tuple[str, str, str]] = []
    for idx, start in enumerate(status_positions):
        end        = status_positions[idx + 1] if idx + 1 < len(status_positions) else len(lines)
        block_text = "\n".join(lines[start:end])
        status     = extract_status(lines[start])
        person = next((name for name, pat in PERSON_TYPES if pat.search(block_text)), "inconnu")
        zone   = next((name for name, pat in ZONES        if pat.search(block_text)), "inconnu")
        tuples.append((person, zone, status))

    # Inférence de zone : si inconnu, hérite du bloc précédent (PDFs DE / EN tronqués)
    for i in range(1, len(tuples)):
        person, zone, status = tuples[i]
        if zone == "inconnu":
            prev_zone = tuples[i - 1][1]
            if prev_zone != "inconnu":
                tuples[i] = (person, prev_zone, status)

    lookup: dict[str, list[str]] = {}
    for person, zone, status in tuples:
        lookup.setdefault(f"{person}-{zone}", []).append(status)

    return lookup

def resolve(lookup: dict, key: str):
    """
    Résout un titre en True / False / 'Error'.
    - Paire absente   → 'Error'
    - Paire ambiguë   → 'Error'
    - Paire unique    → True ou False selon le statut
    """
    vals = lookup.get(key, [])
    if len(vals) != 1:
        return "Error"
    return STATUS_TO_BOOL.get(vals[0], "Error")

# ── Pipeline complet pour un PDF ─────────────────────────────────────────────
def process_pdf(pdf_path: Path) -> dict:
    """Retourne un dict avec une ligne par PDF."""
    ERROR_ROW = {
        "pdf_name": pdf_path.name,
        "prohibition to EEA": "Error",
        "prohibition to UK": "Error",
        "prohibition to Swiss": "Error",
        "pdf_language": "Unknown",
        "content": "",
        "raw_content": "",
    }

    with pdfplumber.open(pdf_path) as pdf:
        pages = [(p.extract_text() or "") for p in pdf.pages]

    # Texte brut complet (toutes pages) — utile pour audit / debug
    raw_full_text = "\n\n".join(pages)

    # Texte de travail (on enlève seulement le bruit connu)
    full_text = cleanup_re.sub("", raw_full_text)

    m_start = start_re.search(full_text)
    if not m_start:
        return {**ERROR_ROW, "raw_content": raw_full_text, "content": "[section DISTRIBUTION introuvable]"}

    m_end = end_re.search(full_text, pos=m_start.end())
    if not m_end:
        return {**ERROR_ROW, "raw_content": raw_full_text, "content": "[section fin introuvable]"}

    filtered = full_text[m_start.start():m_end.start()].strip()

    m_int = interdiction_re.search(filtered)
    if not m_int:
        return {**ERROR_ROW, "raw_content": raw_full_text, "content": "[interdiction introuvable]"}

    language = detect_language(m_int)
    filtered = filtered[m_int.start():].strip()
    filtered = "\n".join(l for l in filtered.splitlines() if not is_full_caps_line(l)).strip()
    filtered = trim_to_last_meaningful(filtered)

    lookup = parse_blocks(filtered)

    return {
        "pdf_name": pdf_path.name,
        "prohibition to EEA": resolve(lookup, "personne-EEE"),
        "prohibition to UK": resolve(lookup, "personne-Royaume-Uni"),
        "prohibition to Swiss": resolve(lookup, "personne-Suisse"),
        "pdf_language": language,
        "content": filtered,
        "raw_content": raw_full_text,
    }

print("Helpers chargés (FR / EN / DE).")


Helpers chargés (FR / EN / DE).


## Cellule 2 — Traitement & export

Cette cellule :
1. Parcourt tous les PDFs du dossier `pdf_to_extract/` (barre de progression incluse)
2. Affiche un aperçu du tableau résultat (sans la colonne `content`, trop longue)
3. Exporte le résultat complet dans **`resultats.csv`**

> **Pour analyser de nouveaux PDFs** : déposer les fichiers dans `pdf_to_extract/` puis ré-exécuter cette cellule uniquement.

In [2]:
# ── Traitement de tous les PDFs du dossier ───────────────────────────────────
from tqdm.notebook import tqdm

pdf_files = sorted(PDF_FOLDER.glob("*.pdf"))

if not pdf_files:
    print(f"Aucun PDF trouvé dans '{PDF_FOLDER}'. Ajoute des fichiers et relance cette cellule.")
else:
    rows = [
        process_pdf(p)
        for p in tqdm(pdf_files, desc="Traitement PDFs", unit="pdf")
    ]
    df = pd.DataFrame(
        rows,
        columns=[
            "pdf_name",
            "prohibition to EEA",
            "prohibition to UK",
            "prohibition to Swiss",
            "pdf_language",
            "content",
            "raw_content",
        ],
    )

    print(f"{len(df)} PDF(s) traité(s).\n")
    display(df.drop(columns=["content", "raw_content"]))  # aperçu sans colonnes texte

    # Parquet (via pyarrow) n'aime pas les colonnes mixtes bool/str.
    # On force donc les colonnes "prohibition" en texte: "True" / "False" / "Error".
    for col in ["prohibition to EEA", "prohibition to UK", "prohibition to Swiss"]:
        df[col] = df[col].astype("string")

    df.to_csv("resultats.csv", index=False, encoding="utf-8-sig")
    print("→ resultats.csv")

    df.to_parquet("resultats.parquet", index=False)
    print("→ resultats.parquet")


Traitement PDFs:   0%|          | 0/28 [00:00<?, ?pdf/s]

28 PDF(s) traité(s).



,pdf_name,prohibition to EEA,prohibition to UK,prohibition to Swiss,pdf_language
0,FT_DE000SH9ZC67.pdf,False,False,False,German
1,FinalTerms_DE000SH9ZC26.pdf,False,False,False,German
2,FinalTerms_DE000SH9ZC34.pdf,False,False,False,German
3,ICH1337324979_F_PC_N.pdf,False,False,False,English
4,ICH1534633511_F_PC_N.pdf,False,False,False,English
5,ICH1534635946_F_PC_N.pdf,False,False,False,English
6,ICH1534635979_F_PC_N.pdf,False,False,False,English
7,ICH1534635987_F_PC_N.pdf,False,False,False,English
8,ICH1534636365_F_PC_N.pdf,False,False,False,English
9,IFRSG00014WC1_F_PC_N.pdf,False,False,False,French


→ resultats.csv
→ resultats.parquet
